In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.15 Waves in Media: Dispersion, Absorption, and the Fresnel Relations

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.15",
    title="Waves in Media: Dispersion, Absorption, and the Fresnel Relations",
    blurb="One driven electron on a spring turns into the refractive index: "
    "its real part refracts, its imaginary part absorbs, its zero is the plasma "
    "frequency that makes metals mirrors, and matching phases across an "
    "interface gives Snell, Fresnel, Brewster, and the evanescent wave.",
    difficulty="advanced",
    estimate="160–190 min",
)

## Notebook overview

[§3.8](maxwell-waves.ipynb) built a wave and set it loose in empty space, where it
travels at $c$ and never changes. [§3.13](fields-in-matter.ipynb) put fields into
matter but held them still. This notebook is where the two meet, and it is the one
both of them promised: a wave arriving in a material, and the material answering back.

The whole of it comes out of a single mechanical picture. An electron bound in an atom
is a mass on a spring with friction, so a passing light wave drives it exactly as the
torque drove the pendulum in
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb). That is not an analogy
offered for comfort: it is the *same* differential equation, and we integrate it here
with the same solver to check it against the same resonance curve. Multiply one
electron's response by the number density and the material's permittivity becomes a
function of frequency, $\varepsilon(\omega)$, complex because the electron lags the
drive. From there everything follows by algebra we can compute rather than quote. The
square root of $\varepsilon$ is a complex refractive index $\tilde n = n + i\kappa$
whose real part slows the wave and whose imaginary part eats it; near the resonance
$n$ *decreases* with frequency, the anomalous dispersion that makes a prism misbehave
where it also absorbs. Set the spring constant to zero and the bound electron becomes a
free one: the Drude conductor, whose absorption length is the **skin depth** (9.2 mm in
copper at 50 Hz, 65 µm at 1 MHz), and whose $\varepsilon$ crosses zero at the **plasma
frequency**. Below that frequency a free-electron gas reflects everything, which is why
aluminium is a mirror up to 15.8 eV and why a 1 MHz radio signal bounces off the
ionosphere while a 100 MHz one is lost to space.

The second half is the interface. A boundary condition has to hold at every point of
the surface at once, and that single requirement forces the three waves to share a
tangential wavevector: **Snell's law is phase matching**, and we show it by measuring
the mismatch as a function of the assumed refraction angle and watching it collapse to
zero at Snell's angle and nowhere else. The amplitudes then come from the jump
conditions of [§3.13](fields-in-matter.ipynb), which for each polarization are two
linear equations in two unknowns: we assemble that $2\times2$ system and solve it, and
out come the **Fresnel relations**. They pay immediate dividends. For one polarization
the reflected amplitude passes through zero at **Brewster's angle**, $56.31^\circ$ for
glass, because there the radiating dipoles in the medium point straight along the
reflected ray and a dipole cannot radiate along its own axis
([§3.10](radiation.ipynb)). Beyond the critical angle the refraction angle stops being
real, the reflection becomes total, and the field that persists on the far side is
**evanescent**: it travels along the surface, decays over 285 nm, is larger in
amplitude than the incident wave, and carries exactly zero energy away. That last
figure is the notebook's one animation, because it is the one place where something
genuinely moves.

Everything is in **SI units**, with the CODATA constants imported from
`scipy.constants`. The material model is a single Lorentz resonance in the ultraviolet
($\hbar\omega_0 = 10$ eV, $\hbar\gamma = 0.5$ eV, bound-electron density
$n_b = 9.07\times10^{28}\,\mathrm{m^{-3}}$), chosen so that its static limit is
$\varepsilon_r(0) = 2.25$, i.e. $n = 1.50$: the index of ordinary glass, out of one
resonance. Every optics exercise from Exercise 6 onward uses that same $n = 1.5$, so
the second half of the notebook is running on the number the first half computed.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: an integrated trajectory against a closed-form complex amplitude,
> $n^2-\kappa^2=\operatorname{Re}\varepsilon_r$, copper's conductivity against its
> measured $5.96\times10^7\,$S/m, aluminium's plasma energy against the measured
> 15.8 eV, $R+T=1$ at every angle of incidence, $|r|=1$ beyond the critical angle. A
> validation compares a result to an expected physical fact, so a ✗ does not by itself
> mean the answer is wrong: it may be a genuine error, a different-but-valid convention
> (a sign, a branch of a square root), or too tight a tolerance. Treat a ✗ as a prompt
> to locate the discrepancy. Passing is strong evidence, not proof.
>
> **A sign-convention note.** We write plane waves as
> $e^{i(\tilde k z-\omega t)}$ throughout. With that choice a passive medium has
> $\operatorname{Im}\varepsilon_r>0$ and $\operatorname{Im}\tilde n\ge0$, which is the
> branch `numpy.sqrt` returns for a complex argument. The opposite convention
> $e^{i(\omega t-\tilde k z)}$, common in engineering, flips the sign of every
> imaginary part; a check that fails only in the sign of $\kappa$ has found the
> convention, not an error.

> **Scope.** A working review, not a full course. See Jackson {cite}`jackson` (ch. 7,
> which is this notebook's chapter); Griffiths, *Introduction to Electrodynamics*
> {cite}`griffiths_em` (ch. 9); Born and Wolf {cite}`bornwolf1999` (ch. 1–2, 13);
> Nolting, *Theoretical Physics 3* {cite}`nolting3`.

## Theory in brief

### The Lorentz oscillator

Bind an electron of charge $-e$ and mass $m$ to its atom with a restoring force
$-m\omega_0^2 x$, let it lose energy at rate $\gamma$, and drive it with the electric
field of a passing wave, $E(t)=E_0\cos\omega t$. Newton's second law reads

```{math}
:label: eq-wm-eom
\ddot x + \gamma\,\dot x + \omega_0^2\,x = -\frac{e}{m}\,E_0\cos\omega t ,
```

which is the damped, driven harmonic oscillator of
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb) with light supplying the
drive. Writing $x(t)=\operatorname{Re}\bigl[\tilde x\,E_0 e^{-i\omega t}\bigr]$, the
steady state is fixed by one complex number,

```{math}
:label: eq-wm-xamp
\tilde x(\omega) \;=\; \frac{-e/m}{\omega_0^2-\omega^2-i\gamma\omega} ,
```

whose modulus is the resonance curve and whose argument fixes the phase by which the
electron lags the force driving it. Since that force is $-eE$, the lag is
$\pi+\arg\tilde x$, and it runs from $0$ well below resonance through exactly $\pi/2$ at
$\omega=\omega_0$ to $\pi$ well above: the electron follows the push at low frequency and
opposes it at high frequency, which is why $\varepsilon_r$ changes sign.

### The dielectric function

Each displaced electron carries a dipole moment $-e\,x$, so a density $n_b$ of them
gives a polarization $P=-e\,n_b x = \varepsilon_0\chi_e E$. Feeding {eq}`eq-wm-xamp`
into $\varepsilon_r = 1+\chi_e$ gives the **dielectric function**

```{math}
:label: eq-wm-epsilon
\varepsilon_r(\omega) \;=\; 1 + \frac{\omega_p^2}{\omega_0^2-\omega^2-i\gamma\omega},
\qquad
\omega_p^2 \;\equiv\; \frac{n_b e^2}{\varepsilon_0 m} ,
```

the frequency-dependent, complex successor to the constant $\varepsilon_r$ of
[§3.13](fields-in-matter.ipynb). It is complex because the electron lags, and a lagging
response means work done on the medium: absorption. Its static value
$\varepsilon_r(0)=1+\omega_p^2/\omega_0^2$ is the ordinary permittivity, and it returns
to $1$ far above resonance, where the electron is too heavy to keep up and the material
might as well be vacuum. Between $\omega_0$ and $\omega_L=\sqrt{\omega_0^2+\omega_p^2}$
a strong resonance drives $\operatorname{Re}\varepsilon_r$ *negative*, a band in which
no wave propagates.

### The complex refractive index

Maxwell's equations in a medium are those of [§3.8](maxwell-waves.ipynb) with
$\varepsilon_0\to\varepsilon_0\varepsilon_r$, so a plane wave $e^{i(\tilde kz-\omega
t)}$ has $\tilde k=\tilde n\,\omega/c$ with

```{math}
:label: eq-wm-index
\tilde n(\omega) \;=\; n(\omega) + i\,\kappa(\omega) \;=\; \sqrt{\varepsilon_r(\omega)},
\qquad n^2-\kappa^2=\operatorname{Re}\varepsilon_r,\quad 2n\kappa=\operatorname{Im}\varepsilon_r .
```

The split is the whole story of optics in one line. The real part $n$ slows the phase
to $c/n$ and bends rays; the imaginary part $\kappa$ (the *extinction coefficient*)
multiplies the field by $e^{-\omega\kappa z/c}$, so the intensity obeys Beer's law

```{math}
:label: eq-wm-beer
I(z)=I_0e^{-\alpha z}, \qquad \alpha = \frac{2\omega\kappa}{c} .
```

Away from resonance $n$ rises with frequency (blue bends more than red: **normal
dispersion**). Inside the absorption line it *falls* — **anomalous dispersion** — and
the two facts are not independent: absorption and dispersion are the imaginary and real
parts of one analytic function, which is the content of the Kramers–Kronig relations
derived in
[§7.2](../07-quantum-statistical-mechanics/complex-analysis-applications.ipynb). We use
nothing from there here; this notebook computes both parts directly from
{eq}`eq-wm-epsilon` and simply observes that they peak together.

### Free electrons: Drude, skin depth, plasma frequency

Take away the spring. With $\omega_0=0$ and $\gamma=1/\tau$ the collision rate,
{eq}`eq-wm-epsilon` becomes the **Drude** dielectric function of a conductor,

```{math}
:label: eq-wm-drude
\varepsilon_r(\omega) \;=\; 1-\frac{\omega_p^2}{\omega(\omega+i/\tau)}
\;\xrightarrow[\ \omega\tau\ll1\ ]{}\; 1 + \frac{i\sigma}{\varepsilon_0\omega},
\qquad \sigma=\frac{n e^2\tau}{m} ,
```

so the conduction current of Ohm's law is what a polarization current becomes when
nothing holds the charge back. At low frequency the second form dominates,
$\tilde n\simeq(1+i)\sqrt{\sigma/2\varepsilon_0\omega}$, and the field falls off inside
the metal over the **skin depth**

```{math}
:label: eq-wm-skin
\delta \;=\; \frac{c}{\omega\kappa} \;\simeq\; \sqrt{\frac{2}{\mu_0\sigma\omega}} ,
```

a millimetre-scale length at mains frequency and a micron at a gigahertz: high-frequency
current rides on the surface of a wire. At the other end, when $\omega\tau\gg1$
collisions are irrelevant and {eq}`eq-wm-drude` collapses to

```{math}
:label: eq-wm-plasma
\varepsilon_r(\omega) \;=\; 1-\frac{\omega_p^2}{\omega^2} ,
```

negative below the **plasma frequency** $\omega_p$ and positive above it. Negative
$\varepsilon_r$ means purely imaginary $\tilde n$: the wave cannot propagate, it decays,
and with no absorption to accompany the decay it is reflected in full. That single sign
change explains the shine of a clean metal, the ultraviolet transparency window of
aluminium, and the fact that the ionosphere is a mirror for the AM band and a window for
FM.

### Snell's law is phase matching

At a flat interface the boundary conditions of [§3.13](fields-in-matter.ipynb) —
$E_\parallel$ and $H_\parallel$ continuous — must hold at **every point of the surface,
at every instant**. The three waves therefore have to agree in their variation along the
surface, which forces one frequency and one tangential wavevector on all of them. With
$k_j=n_j\omega/c$ that is

```{math}
:label: eq-wm-snell
\theta_r=\theta_i, \qquad n_1\sin\theta_i = n_2\sin\theta_t ,
```

the law of reflection and Snell's law, obtained without ever writing an amplitude.

### The Fresnel relations

The amplitudes come next, from the same two continuity conditions, and they depend on
polarization. For **s**-polarization ($\mathbf E$ perpendicular to the plane of
incidence) continuity of $E_\parallel$ and of $H_\parallel$ reads $1+r_s=t_s$ and
$n_1\cos\theta_i(1-r_s)=n_2\cos\theta_t\,t_s$, whose solution is

```{math}
:label: eq-wm-fresnel-s
r_s=\frac{n_1\cos\theta_i-n_2\cos\theta_t}{n_1\cos\theta_i+n_2\cos\theta_t},
\qquad
t_s=\frac{2n_1\cos\theta_i}{n_1\cos\theta_i+n_2\cos\theta_t} .
```

For **p**-polarization ($\mathbf E$ in the plane of incidence) the same two conditions
read $n_1(1+r_p)=n_2t_p$ and $\cos\theta_i(1-r_p)=\cos\theta_t\,t_p$, giving

```{math}
:label: eq-wm-fresnel-p
r_p=\frac{n_2\cos\theta_i-n_1\cos\theta_t}{n_2\cos\theta_i+n_1\cos\theta_t},
\qquad
t_p=\frac{2n_1\cos\theta_i}{n_2\cos\theta_i+n_1\cos\theta_t} .
```

The measured quantities are the reflectance $R=|r|^2$ and the transmittance
$T=(n_2\cos\theta_t/n_1\cos\theta_i)|t|^2$, whose geometric prefactor accounts for the
beam's change of cross-section and of speed; energy conservation is the statement
$R+T=1$.

### Brewster's angle, total internal reflection, the evanescent wave

The numerator of $r_p$ vanishes when $n_2\cos\theta_i=n_1\cos\theta_t$, which with
{eq}`eq-wm-snell` happens at **Brewster's angle**

```{math}
:label: eq-wm-brewster
\tan\theta_B = \frac{n_2}{n_1}, \qquad\text{equivalently}\qquad \theta_B+\theta_t=90^\circ .
```

The second form is the physical one: at $\theta_B$ the dipoles driven in the second
medium oscillate *along* the direction the reflected ray would take, and a dipole
radiates nothing along its own axis ([§3.10](radiation.ipynb)). Reflected light at
$\theta_B$ is purely s-polarized, which is what polarizing sunglasses exploit.

Going the other way, from glass into air, {eq}`eq-wm-snell` has no real solution once
$\sin\theta_i>n_2/n_1$. Past that **critical angle** $\theta_c=\arcsin(n_2/n_1)$ we set
$\cos\theta_t=i\,q$ with $q=\sqrt{(n_1\sin\theta_i/n_2)^2-1}$, and the transmitted wave
becomes **evanescent**,

```{math}
:label: eq-wm-evanescent
E_t \propto e^{\,i k_1\sin\theta_i\,x}\,e^{-z/d},
\qquad d = \frac{\lambda}{2\pi\sqrt{n_1^2\sin^2\theta_i-n_2^2}} ,
```

travelling along the surface and dying within a fraction of a wavelength of it. Both
Fresnel coefficients then have modulus one — the reflection is *total* — while $|t|$ can
exceed one, because the evanescent field stores energy near the surface without carrying
any away. Frustrate that decay with a second piece of glass a few hundred nanometres off
and the light crosses the gap: the optical analogue of quantum tunnelling, and the
working principle of every fingerprint sensor and near-field microscope.

## Setup

Data and instruments only: the CODATA constants, the parameters of the model
dielectric and of the three real materials the notebook grades itself against (copper,
aluminium, the night-time ionosphere), the optical bench of a helium–neon wavelength
and the two indices $n=1$ and $n=1.5$, the series palette, and one unit conversion
between photon energy in electronvolts and angular frequency in rad/s. Nothing in the
notebook's own line of argument is pre-built: the equation of motion, the dielectric
function, the complex index, the Drude limit, the phase-matching mismatch and the
$2\times2$ boundary-condition system that yields the Fresnel coefficients are all
assembled in the exercises. Nothing here is stochastic.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

# data: CODATA constants — the vacuum pair that fixes c, and the charge and mass of the
# electron, which is all a Lorentz oscillator needs
from scipy.constants import c as C0  # speed of light in vacuum, m/s
from scipy.constants import e as QE  # elementary charge, C
from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m
from scipy.constants import eV  # one electronvolt, J
from scipy.constants import hbar as HBAR  # reduced Planck constant, J·s
from scipy.constants import m_e as ME  # electron mass, kg
from scipy.constants import mu_0 as MU0  # vacuum permeability, T·m/A

# data: the series palette
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT

# data: the model dielectric — one Lorentz resonance in the ultraviolet. The bound
# electron density is chosen so that eps_r(0) = 1 + wp^2/w0^2 = 2.25, i.e. n(0) = 1.50,
# the index of ordinary glass that the second half of the notebook then works with.
E_RES_EV = 10.0  # resonance energy ħω₀, eV
E_DAMP_EV = 0.5  # damping ħγ, eV
N_BOUND = 9.07e28  # bound-electron density n_b, m⁻³

# data: copper at room temperature — conduction-electron density and relaxation time,
# with the measured DC conductivity they are graded against
N_CU = 8.47e28  # conduction electrons per m³
TAU_CU = 2.5e-14  # Drude relaxation time, s
SIGMA_CU_MEAS = 5.96e7  # measured DC conductivity, S/m

# data: aluminium — three valence electrons per atom, and the measured bulk plasmon energy
N_AL = 1.808e29  # conduction electrons per m³
TAU_AL = 8.0e-15  # Drude relaxation time, s
E_PLASMA_AL_MEAS = 15.8  # measured bulk plasmon energy ħω_p, eV

# data: the night-time ionospheric F layer
N_IONO = 1.0e11  # free-electron density, m⁻³

# data: the optical bench — a helium–neon wavelength, and the two indices used from
# Exercise 6 onward (the glass value is the static limit of the model dielectric)
LAMBDA_HENE = 633e-9  # vacuum wavelength, m
N_AIR, N_GLASS = 1.0, 1.5


# instrument: one unit conversion between photon energy and angular frequency. Optical
# and solid-state data are quoted in eV while every formula above wants rad/s; converting
# between them is bookkeeping, not the lesson of any exercise.
def omega_from_eV(energy_eV):
    """Angular frequency corresponding to a photon energy given in electronvolts.

    Physically the single relation ω = E/ħ, applied so that resonance energies,
    plasma energies and spectra can be quoted the way they are measured (eV)
    while every dielectric formula is evaluated in SI (rad/s).

    Parameters
    ----------
    energy_eV : float or numpy.ndarray
        Photon energy in electronvolts.

    Returns
    -------
    float or numpy.ndarray
        Angular frequency in rad/s, of the same shape as the input.
    """
    return np.asarray(energy_eV) * eV / HBAR

## Exercise 1 — The Lorentz oscillator: the driven oscillator inside an atom (worked)

An electron bound in an atom is a mass on a spring. Pull it a distance $x$ off centre
and the rest of the atom pulls back with $-m\omega_0^2x$; let it radiate and collide and
it loses energy at a rate $\gamma$; send a light wave past and its electric field pushes
on the charge $-e$. Newton's second law is then {eq}`eq-wm-eom`, and that equation is
not merely *like* the damped driven pendulum of
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb) in the small-angle
regime: it is the same equation, with $\ell$ and $g$ replaced by an atom and light
replacing the shaker. Everything that notebook found — a resonance peak, a phase lag
growing from $0$ through $-\pi/2$ to $-\pi$, a width set by the damping — is about to
become the optics of glass ({numref}`fig-wm-lorentz-setup`).

The steady state is one complex number, {eq}`eq-wm-xamp`. Its modulus gives the
amplitude and its argument the lag, and both are cleanly measurable from an integrated
trajectory: writing the late-time motion as $x(t)=\operatorname{Re}[X e^{-i\omega t}]$
means $x(t)=\operatorname{Re}X\,\cos\omega t+\operatorname{Im}X\,\sin\omega t$, so
projecting the sampled $x(t)$ onto $\cos\omega t$ and $\sin\omega t$ over a whole number
of drive periods returns $X$ itself.

The oscillator here is the model dielectric's: $\hbar\omega_0=10\,$eV,
$\hbar\gamma=0.5\,$eV, driven by $E_0=10^{5}\,$V/m (an ordinary laboratory field, well
inside the linear regime).

**Part a)** Write `lorentz_rhs(t, s, omega_d)` for the state $s=(x,\dot x)$, returning
$(\dot x,\;-\omega_0^2x-\gamma\dot x-(e/m)E_0\cos\omega_d t)$ from {eq}`eq-wm-eom`, with
$e$ and $m$ the CODATA `QE` and `ME`. **Write this one yourself** — the implementation
is the lesson.

**Part b)** Integrate it from rest, $x(0)=\dot x(0)=0$, at the drive frequency
$\omega_d=0.8\,\omega_0$ for 60 drive periods with
`scipy.integrate.solve_ivp(method="DOP853", rtol=1e-10, atol=1e-30)` — a high-order
explicit Runge–Kutta at tight tolerance, because the quantity of interest is a phase
accumulated over sixty cycles — sampling the **last ten periods** on a
`t_eval` grid of 20001 points so the transient is gone and the projections of Part c
have a dense, whole-period window.

**Part c)** Extract the measured complex amplitude by the two projections
$\operatorname{Re}X=(2/T_{\rm win})\int x\cos\omega_d t\,dt$ and
$\operatorname{Im}X=(2/T_{\rm win})\int x\sin\omega_d t\,dt$, each evaluated with
`numpy.trapezoid` over the ten-period window $T_{\rm win}$, and compare $X$ with
$\tilde x(\omega_d)E_0$ from {eq}`eq-wm-xamp`.

**Part d)** Repeat the integration at 15 drive frequencies spaced linearly over
$\omega_d/\omega_0\in[0.2,\,1.8]$ and plot the measured amplitude $|X|$ and the lag
behind the driving force, $\pi+\arg X$, against the closed-form curves
$|\tilde x(\omega)|E_0$ and $\pi+\arg\tilde x(\omega)$
({numref}`fig-wm-lorentz-response`). The lag passes through exactly $\pi/2$ at
$\omega_d=\omega_0$, where the denominator of {eq}`eq-wm-xamp` is purely imaginary.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    abs(X_meas),
    abs(X_exact),
    "the integrated steady-state amplitude at ω_d = 0.8ω₀ matches eq-wm-xamp",
    rtol=1e-3,
)
validate.close(
    np.angle(X_meas),
    np.angle(X_exact),
    "so does its phase arg X, which fixes how far the electron trails the field",
    rtol=1e-3,
)
validate.close(
    np.abs(X_sweep),
    np.abs(X_sweep_exact),
    "the swept resonance curve reproduces |x̃(ω)|E₀ across 0.2 ω₀ to 1.8 ω₀",
    rtol=1e-2,
)
validate.close(
    lag_res,
    np.pi / 2,
    "the lag behind the driving force is exactly π/2 on resonance",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — From one electron to $\varepsilon(\omega)$ (student)

One oscillator is a curiosity; a cubic metre of them is a material. An electron
displaced by $x$ carries a dipole moment $-ex$, so a density $n_b$ of identical bound
electrons gives a polarization $P=-e\,n_b\,x$, and with the steady state
{eq}`eq-wm-xamp` that polarization is proportional to the field it responds to. Reading
off the susceptibility from $P=\varepsilon_0\chi_e E$ and adding the vacuum term gives
the **dielectric function** {eq}`eq-wm-epsilon`, with the combination
$\omega_p^2=n_be^2/\varepsilon_0m$ collecting the density, charge and mass into a single
frequency. The constant $\varepsilon_r$ that [§3.13](fields-in-matter.ipynb) worked with
is now a curve, and a complex one.

Three of its features are worth locating rather than admiring. Its static value
$\varepsilon_r(0)=1+\omega_p^2/\omega_0^2$ is the permittivity a capacitor would
measure. Far above resonance the binding stops mattering at all, since
$\omega^2\gg\omega_0^2$ makes the denominator of {eq}`eq-wm-epsilon` into $-\omega^2$
and leaves $\varepsilon_r\to1-\omega_p^2/\omega^2$: a *free*-electron response, which is
exactly the plasma dielectric function {eq}`eq-wm-plasma` that Exercise 5 is about. The
approach to the vacuum value $1$ is therefore slow, from below. In between, if the
resonance is strong enough, $\operatorname{Re}\varepsilon_r$ dips below zero over a band, ending at the zero
$\omega_L=\sqrt{\omega_0^2+\omega_p^2}$ — a band where the wave cannot propagate at all,
and the same zero that will reappear in Exercise 5 as the plasma frequency once
$\omega_0$ is set to zero.

The material is the model dielectric of Exercise 1: $\hbar\omega_0=10\,$eV,
$\hbar\gamma=0.5\,$eV, $n_b=9.07\times10^{28}\,\mathrm{m^{-3}}$.

**Part a)** Build $\omega_p^2=n_be^2/(\varepsilon_0 m)$ from the Setup constants, then
evaluate {eq}`eq-wm-epsilon` as an explicit complex `numpy` array on 4001 photon
energies spaced linearly over $\hbar\omega\in[0.05,\,30]\,$eV (converted to rad/s with
`omega_from_eV`), and plot $\operatorname{Re}\varepsilon_r$ and
$\operatorname{Im}\varepsilon_r$ ({numref}`fig-wm-epsilon`).

**Part b)** Report $\varepsilon_r$ at the lowest grid energy against the static value
$1+\omega_p^2/\omega_0^2$, and at $30\,$eV against the free-electron form
$1-\omega_p^2/\omega^2$.

**Part c)** Locate the zero of $\operatorname{Re}\varepsilon_r$ above the resonance with
`scipy.optimize.brentq` bracketed on $[11,\,20]\,$eV, and compare it with
$\hbar\omega_L=\hbar\sqrt{\omega_0^2+\omega_p^2}$. The small offset between them is the
damping's doing, and it shrinks with $\gamma$.

**Part d)** Report the energy band on which $\operatorname{Re}\varepsilon_r<0$ (a
`numpy` boolean mask over the grid), and confirm with `numpy.all` that
$\operatorname{Im}\varepsilon_r>0$ everywhere on the grid — a passive medium absorbs
energy at every frequency and supplies it at none.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.real(eps_model[0]),
    eps_static,
    "the static limit of ε_r(ω) is 1 + ω_p²/ω₀² = 2.2506, the permittivity of §3.13",
    rtol=1e-4,
)
validate.close(
    np.real(eps_model[-1]),
    eps_free_30,
    "far above resonance the binding is irrelevant and ε_r → 1 − ω_p²/ω², the free-electron form of Exercise 5",
    rtol=3e-2,
)
validate.close(
    w_zero,
    wL_exact,
    "Re ε_r crosses zero at ω_L = √(ω₀² + ω_p²), the top of the no-propagation band",
    rtol=5e-3,
)
validate.check(
    passive,
    "Im ε_r > 0 at every frequency: the medium absorbs energy and never supplies it",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — The complex refractive index, and anomalous dispersion (student)

Maxwell's equations inside a linear medium are those of [§3.8](maxwell-waves.ipynb) with
$\varepsilon_0$ replaced by $\varepsilon_0\varepsilon_r(\omega)$, so a plane wave
$e^{i(\tilde kz-\omega t)}$ travels with $\tilde k=\tilde n\,\omega/c$ and a **complex**
refractive index $\tilde n=\sqrt{\varepsilon_r}$, {eq}`eq-wm-index`. Splitting it as
$\tilde n=n+i\kappa$ splits the physics with it: the factor $e^{i\omega nz/c}$ is a wave
whose phase velocity is $c/n$, and the factor $e^{-\omega\kappa z/c}$ is a decay.
Intensity goes as the modulus squared, hence Beer's law {eq}`eq-wm-beer` with absorption
coefficient $\alpha=2\omega\kappa/c$. One square root of one complex function has
produced both of the things a material does to light.

Squaring $\tilde n=n+i\kappa$ ties the two halves to the two halves of
$\varepsilon_r$: $n^2-\kappa^2=\operatorname{Re}\varepsilon_r$ and
$2n\kappa=\operatorname{Im}\varepsilon_r$. Those identities are the sharp test that the
right branch of the square root has been taken, since $\sqrt{\varepsilon_r}$ has two
values and only the one with $\kappa\ge0$ describes a wave that decays as it goes.

The behaviour worth hunting for is **anomalous dispersion**. Outside an absorption line
$n$ increases with frequency, which is why a prism bends blue more than red; inside the
line $n$ *falls*, so briefly the prism runs backwards. The two effects are not
independent, because $\operatorname{Re}\varepsilon_r$ and $\operatorname{Im}\varepsilon_r$
are the real and imaginary parts of one analytic function — the Kramers–Kronig statement
proved in
[§7.2](../07-quantum-statistical-mechanics/complex-analysis-applications.ipynb). Nothing
here depends on that result; we simply compute both parts from {eq}`eq-wm-epsilon` and
find them peaked together.

The material is again the model dielectric ($\hbar\omega_0=10\,$eV, $\hbar\gamma=0.5\,$eV,
$n_b=9.07\times10^{28}\,\mathrm{m^{-3}}$), on the same $\hbar\omega\in[0.05,\,30]\,$eV
grid of 4001 points built in Exercise 2.

**Part a)** Form $\tilde n=$ `numpy.sqrt` of the complex array `eps_model` of Exercise 2
(`numpy` returns the principal branch, the one with $\operatorname{Im}\tilde n\ge0$), take
`.real` and `.imag` for $n$ and $\kappa$, and plot both ({numref}`fig-wm-index`).

**Part b)** Confirm the branch with `numpy.allclose` on the two identities of
{eq}`eq-wm-index`, $n^2-\kappa^2=\operatorname{Re}\varepsilon_r$ and
$2n\kappa=\operatorname{Im}\varepsilon_r$, and report $n$ and $\kappa$ at the lowest grid
energy: $n\to\sqrt{2.2506}=1.500$, the glass index the rest of the notebook uses, and
$\kappa\to0$.

**Part c)** Compute $\alpha=2\omega\kappa/c$ from {eq}`eq-wm-beer` and report the
absorption length $1/\alpha$ at $\hbar\omega=2\,$eV (visible light, far below the line)
and at $\hbar\omega=\hbar\omega_0=10\,$eV (dead centre).

**Part d)** Differentiate $n(\omega)$ with `numpy.gradient` on the $\omega$ grid and mask
where $dn/d\omega<0$: that band is the anomalous dispersion. Confirm it is one contiguous
interval (`numpy.diff` of the masked indices is all ones), that the absorption peak
$\arg\max\operatorname{Im}\varepsilon_r$ lies inside it, and that $dn/d\omega>0$ at both
$2\,$eV and $25\,$eV, where dispersion is normal.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    branch_a and branch_b,
    "the square root took the physical branch: n² − κ² = Re ε_r and 2nκ = Im ε_r",
)
validate.close(
    n_re[0],
    np.sqrt(eps_static),
    "the static index is √ε_r(0) = 1.500 — the glass value used from Exercise 6 on",
    rtol=1e-4,
)
validate.close(
    kappa[0],
    0.0,
    "and κ → 0 there: far below resonance the medium is transparent",
    atol=1e-3,
)
validate.check(
    contiguous and peak_inside and normal_outside,
    "anomalous dispersion (dn/dω < 0) occupies one band that contains the absorption "
    "peak, with normal dispersion on both sides of it",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Free electrons, absorption, and the skin depth (student)

Now take the spring away. In a metal the conduction electrons are bound to no particular
atom, so $\omega_0=0$ and the only thing opposing the drive is the scattering that gives
a relaxation time $\tau$, i.e. $\gamma=1/\tau$. Putting those into {eq}`eq-wm-epsilon`
gives the **Drude** dielectric function {eq}`eq-wm-drude`. Expanding it for
$\omega\tau\ll1$ leaves $\varepsilon_r\simeq i\sigma/\varepsilon_0\omega$ with
$\sigma=ne^2\tau/m$, which is worth pausing on: Ohm's law is not a separate piece of
physics bolted onto the theory, it is what the polarization response of Exercise 2
becomes when nothing holds the charge in place ({numref}`fig-wm-skin-setup`).

A purely imaginary $\varepsilon_r$ has $\tilde n=(1+i)\sqrt{\sigma/2\varepsilon_0\omega}$,
equal real and imaginary parts, so a metal at low frequency attenuates a wave over the
same distance in which it turns it through a radian. That distance is the **skin depth**
{eq}`eq-wm-skin`, $\delta=c/\omega\kappa\simeq\sqrt{2/\mu_0\sigma\omega}$, and it is the
reason a mains busbar is solid while a radio-frequency conductor is a thin plated tube:
above a few megahertz the current rides in a skin microns thick and the interior of the
metal is dead weight.

The metal here is **copper**, with $n=8.47\times10^{28}\,\mathrm{m^{-3}}$ conduction
electrons and $\tau=2.5\times10^{-14}\,$s, graded against the measured DC conductivity
$\sigma=5.96\times10^{7}\,$S/m.

**Part a)** Evaluate $\sigma=ne^2\tau/m$ and $\omega_p^2=ne^2/\varepsilon_0m$ as scalar
arithmetic expressions in the Setup values `N_CU`, `TAU_CU` and the CODATA `QE`, `ME`,
`EPS0`, and compare $\sigma$ with the measured $5.96\times10^{7}\,$S/m. Report
$\hbar\omega_p$ in eV (copper's is close to $10.8\,$eV).

**Part b)** Write `eps_drude(omega, wp2, tau)` returning the **full** Drude dielectric
function {eq}`eq-wm-drude`, $1-\omega_p^2/\bigl(\omega(\omega+i/\tau)\bigr)$, as a
complex `numpy` expression. Exercise 5 reuses it in the collisionless limit.

**Part c)** At $f=50\,$Hz, $1\,$MHz and $1\,$GHz, take
$\kappa=\operatorname{Im}\sqrt{\varepsilon_r}$ from it and form
$\delta_{\rm Drude}=c/\omega\kappa$; tabulate it beside the good-conductor formula
$\delta=\sqrt{2/\mu_0\sigma\omega}$ of {eq}`eq-wm-skin`.

**Part d)** Sweep $f$ over 400 points on a `numpy.logspace` from $1\,$Hz to
$10^{16}\,$Hz, plot both expressions ({numref}`fig-wm-skin-depth`), and locate the
frequency at which they first differ by more than $1\%$ using a `numpy` boolean mask on
the relative difference. The good-conductor approximation is not a crude one: it holds
over thirteen decades.

**Part e)** Evaluate both at the helium–neon wavelength $\lambda=633\,$nm
($\omega=2\pi c/\lambda$) and report the ratio. Optical frequencies are the regime
$\omega\tau\gtrsim1$, where the approximation has stopped being an approximation at all.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    SIGMA_CU,
    SIGMA_CU_MEAS,
    "copper's Drude conductivity ne²τ/m reproduces the measured 5.96×10⁷ S/m",
    rtol=2e-2,
)
validate.close(
    delta_drude_table[:2],
    delta_good_table[:2],
    "the full Drude skin depth equals √(2/μ₀σω) at 50 Hz and 1 MHz, where ωτ ≪ 1",
    rtol=1e-4,
)
validate.close(
    delta_good_table[0],
    9.2e-3,
    "the skin depth of copper at mains frequency is 9.2 mm",
    rtol=1e-2,
)
validate.close(
    delta_good_table[1],
    65e-6,
    "and 65 µm at 1 MHz: high-frequency current rides on the surface",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The plasma frequency: why metals shine and the ionosphere bounces radio (student)

Push the Drude function {eq}`eq-wm-drude` to the opposite extreme. When $\omega\tau\gg1$
the electron completes many cycles between collisions, damping becomes irrelevant, and
what remains is {eq}`eq-wm-plasma`, $\varepsilon_r=1-\omega_p^2/\omega^2$: real, no
absorption, and *negative* for every $\omega<\omega_p$. A negative $\varepsilon_r$ gives
a purely imaginary $\tilde n=i\kappa$, so no wave propagates. Yet with $n=0$ there is
nothing to absorb either, and the energy that cannot go forward has only one place left
to go. **The metal reflects it all.**

The mechanical picture behind $\omega_p$ is worth having ({numref}`fig-wm-plasma-setup`).
Displace the whole electron sea rigidly by $u$ against its fixed positive background: one
face of the slab is left with an uncompensated positive sheet, the other with an excess
negative one, and the field between them, $E=n e u/\varepsilon_0$, pulls the electrons
back. That is a harmonic restoring force with $\ddot u=-(ne^2/\varepsilon_0 m)u$, so the
gas rings at exactly $\omega_p$. The plasma frequency is not an abstraction fitted to a
curve, it is the natural frequency of the electron gas as a whole.

Two materials make the point. **Aluminium**, with three valence electrons per atom
($n=1.808\times10^{29}\,\mathrm{m^{-3}}$, $\tau=8\,$fs), has its $\omega_p$ in the deep
ultraviolet, so it is a mirror across the entire visible spectrum and turns transparent
only above $\sim16\,$eV; the measured bulk plasmon energy is $15.8\,$eV. The
**night-time ionosphere**, a far thinner gas at
$n_e=1\times10^{11}\,\mathrm{m^{-3}}$, has its $\omega_p$ down in the megahertz, which is
why an AM broadcast at 1 MHz reflects off the sky and can be heard a continent away while
an FM signal at 100 MHz goes straight through to the satellites.

**Part a)** Compute $\omega_p=\sqrt{ne^2/\varepsilon_0m}$ for aluminium and report
$\hbar\omega_p$ in eV against the measured $15.8\,$eV, and the corresponding vacuum
wavelength $\lambda_p=2\pi c/\omega_p$.

**Part b)** With the `eps_drude` you wrote in Exercise 4, at $\tau=8\,$fs, evaluate the
normal-incidence reflectance $R=\bigl|(1-\tilde n)/(1+\tilde n)\bigr|^2$ — the
$\theta_i=0$ case of {eq}`eq-wm-fresnel-s`, where the two polarizations coincide — on 1200
photon energies spaced linearly over $[1,\,30]\,$eV, and plot it
({numref}`fig-wm-plasma-reflectance`).

**Part c)** Repeat with the collisionless {eq}`eq-wm-plasma` and confirm with
`numpy.allclose` that $R=1$ *exactly* (to $10^{-12}$) at every energy below
$\hbar\omega_p$, and $R<1$ above. Reflection without absorption is what a negative
$\varepsilon_r$ means.

**Part d)** For the ionosphere compute $f_p=\omega_p/2\pi$ and compare it with the
radio engineer's rule $f_p=8.98\sqrt{n_e}\,$Hz ($n_e$ in $\mathrm{m^{-3}}$), then classify
a 1 MHz AM carrier and a 100 MHz FM carrier as reflected or transmitted by evaluating the
sign of $\varepsilon_r$ at each.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    E_plasma_al,
    E_PLASMA_AL_MEAS,
    "aluminium's computed plasma energy ħ√(ne²/ε₀m) matches the measured 15.8 eV",
    rtol=1e-2,
)
validate.check(
    total_below and partial_above,
    "a collisionless electron gas reflects perfectly (R = 1 to 1e-12) below ω_p and "
    "transmits above it: negative ε_r means reflection without absorption",
)
validate.close(
    f_p_iono,
    f_p_rule,
    "the ionospheric plasma frequency agrees with the radio rule f_p = 8.98√n_e Hz",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Snell's law is phase matching (worked)

Everything so far has happened inside one uniform material. Put two of them face to face
and the question becomes what a wave does at the boundary ({numref}`fig-wm-interface`).
The answer divides cleanly in two, and the first half needs no amplitudes at all.

The boundary conditions of [§3.13](fields-in-matter.ipynb) — $E_\parallel$ continuous,
$H_\parallel$ continuous — are not conditions at one point. They must hold at **every**
point $x$ of the interface and at **every** time $t$. The incident, reflected and
transmitted waves carry the phase factors $e^{i(k_{1x}x-\omega_i t)}$,
$e^{i(k_{rx}x-\omega_r t)}$ and $e^{i(k_{tx}x-\omega_t t)}$, and a sum of such factors can
vanish for all $x$ and all $t$ only if the factors are identical. Hence one frequency for
all three, and one tangential wavevector: $k_1\sin\theta_i=k_1\sin\theta_r=k_2\sin\theta_t$
with $k_j=n_j\omega/c$, which is {eq}`eq-wm-snell`. Reflection angle equals incidence
angle, and $n_1\sin\theta_i=n_2\sin\theta_t$. Snell's law is not an empirical rule about
rays; it is what "for all $x$" means.

That argument can be *measured* rather than asserted. Take the incident wave's variation
along the interface, $e^{ik_1\sin\theta_i x}$, and a trial transmitted variation
$e^{ik_2\sin\theta_t x}$ at an assumed angle $\theta_t$, and compute how badly they fail to
match over a stretch of the surface,

$$
M(\theta_t) \;=\; \frac{1}{L}\int_0^{L}\Bigl|e^{ik_1\sin\theta_i x}-e^{ik_2\sin\theta_t x}\Bigr|^2dx .
$$

For any $\theta_t$ but one, the two run out of step and $M$ is of order $2$; at Snell's
angle it is exactly zero, and stays zero however long the interface.

The interface here is **air to glass**, $n_1=1$ and $n_2=1.5$ (the static index computed in
Exercise 3), at vacuum wavelength $\lambda=633\,$nm and incidence angle
$\theta_i=40^\circ$; the matching stretch is $L=20\lambda$.

**Part a)** Build $M(\theta_t)$ with `numpy.trapezoid` on 20001 sample points across
$[0,L]$, evaluate it on 1801 trial angles spaced linearly over $[0^\circ,89.9^\circ]$, and
plot it ({numref}`fig-wm-phase-matching`). Report $M$ at the minimum and at
$\theta_t\pm2^\circ$.

**Part b)** Locate the transmitted angle as the root of the tangential-wavevector mismatch
$g(\theta_t)=k_1\sin\theta_i-k_2\sin\theta_t$ with `scipy.optimize.brentq` bracketed on
$[0,\pi/2]$ (the function is monotone there, so the bracket is safe), and compare it with
$\arcsin(n_1\sin\theta_i/n_2)=25.374^\circ$.

**Part c)** Apply the same matching within medium 1, where the reflected wave has the same
$|k|$ as the incident one, so $\sin\theta_r=\sin\theta_i$; confirm the reflected angle
equals $40^\circ$.

**Part d)** Reverse the media to glass into air. Show that $g$ has no root once
$\sin\theta_i>n_2/n_1$ (evaluate its sign at both bracket ends), and locate the **critical
angle** as the root of $n_1\sin\theta-n_2$ with `scipy.optimize.brentq` on
$[0,\pi/2]$: $\theta_c=\arcsin(n_2/n_1)=41.810^\circ$, the gateway to Exercise 9.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    th_t_root,
    TH_T_DEMO,
    "phase matching gives Snell's law: the mismatch root is arcsin(n₁sinθ_i/n₂) = 25.374°",
    rtol=1e-9,
)
validate.close(
    th_r_root, TH_I_DEMO, "and the law of reflection, θ_r = θ_i = 40°", rtol=1e-9
)
validate.close(
    M_snell,
    0.0,
    "the interface mismatch vanishes at Snell's angle",
    atol=1e-12,
)
validate.check(
    min(M_off) > 0.1,
    "and is of order unity two degrees away: only one angle matches phases along the surface",
    f"M(θ_t ± 2°) = {M_off[0]:.3f}, {M_off[1]:.3f}",
)
validate.check(no_root, "glass into air at 60° has no real refraction angle at all")
validate.close(
    th_c,
    np.arcsin(N_AIR / N_GLASS),
    "the critical angle for glass into air is arcsin(n₂/n₁) = 41.810°",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The Fresnel relations, from two continuity conditions (student)

Phase matching fixed the *directions*; the amplitudes need the boundary conditions
themselves. With no free surface charge and no free surface current, the conditions of
[§3.13](fields-in-matter.ipynb) say that the components of $\mathbf E$ and of $\mathbf H$
lying *in* the interface are continuous across it. Two conditions, two unknowns (the
reflected and transmitted amplitudes, in units of the incident one), and the answer
depends on which way the field points relative to the plane of incidence — the plane
containing the incident ray and the normal, drawn in {numref}`fig-wm-interface`.

For **s-polarization** $\mathbf E$ is perpendicular to that plane, so it lies entirely in
the interface and continuity of $E_\parallel$ reads $1+r_s=t_s$; the accompanying
$\mathbf H$ lies in the plane of incidence, its tangential component picks up a
$\cos\theta$, and with $H=nE/\mu_0c$ its continuity reads
$n_1\cos\theta_i\,(1-r_s)=n_2\cos\theta_t\,t_s$ (the minus sign because the reflected
wave's $\mathbf H$ has reversed). For **p-polarization** the roles swap: $\mathbf H$ is
perpendicular to the plane of incidence and gives $n_1(1+r_p)=n_2t_p$, while $\mathbf E$
lies in it and gives $\cos\theta_i\,(1-r_p)=\cos\theta_t\,t_p$. Each pair is a
$2\times2$ linear system, and solving it is the derivation of the Fresnel relations
{eq}`eq-wm-fresnel-s` and {eq}`eq-wm-fresnel-p`.

What a detector reports is not the amplitude but the power. The reflectance is
$R=|r|^2$, while the transmittance carries a geometric factor,
$T=(n_2\cos\theta_t/n_1\cos\theta_i)\,|t|^2$, because the refracted beam has a different
cross-section and travels at a different speed. Energy conservation is then the single
statement $R+T=1$, which the coefficients must satisfy at every angle without having been
asked to.

The interface is again **air to glass**, $n_1=1$, $n_2=1.5$.

**Part a)** Write `fresnel_s(n1, n2, theta_i)`: form $\cos\theta_t$ from
{eq}`eq-wm-snell` as `numpy.sqrt(1 - (n1*sin(theta_i)/n2)**2 + 0j)` (complex, so that
Exercise 9 can reuse it unchanged past the critical angle), assemble the $2\times2$
coefficient matrix and right-hand side of the two s-polarization conditions above as an
explicit `numpy` array, and solve with `numpy.linalg.solve` for $(r_s,t_s)$. Writing the
rows means deciding which condition supplies which row and where each $\cos\theta$ and
each $n$ belongs. **Write this one yourself** — the implementation is the lesson.

**Part b)** The same for `fresnel_p(n1, n2, theta_i)` from the p-polarization pair.

**Part c)** Evaluate both on 1801 angles spaced linearly over $[0^\circ,89.9^\circ]$ and
compare with the closed forms {eq}`eq-wm-fresnel-s` and {eq}`eq-wm-fresnel-p` using
`numpy.allclose`.

**Part d)** Form $R=|r|^2$ and $T=(n_2\cos\theta_t/n_1\cos\theta_i)|t|^2$ for both
polarizations, plot them ({numref}`fig-wm-fresnel`), confirm $R+T=1$ at every angle, and
report the normal-incidence value against $((n_1-n_2)/(n_1+n_2))^2=0.04$: a window pane
reflects four percent per surface, which is why a shop window at night is a mirror.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    matches_closed,
    "the 2×2 boundary-condition system reproduces the Fresnel relations at every angle",
)
validate.close(
    R_s + T_s,
    np.ones_like(R_s),
    "energy is conserved for s-polarization: R + T = 1 at every angle of incidence",
    rtol=1e-12,
)
validate.close(
    R_p + T_p,
    np.ones_like(R_p),
    "and for p-polarization",
    rtol=1e-12,
)
validate.close(
    R_s[0],
    ((N_AIR - N_GLASS) / (N_AIR + N_GLASS)) ** 2,
    "at normal incidence both polarizations give R = ((n₁−n₂)/(n₁+n₂))² = 0.04",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Brewster's angle (student)

One line of {eq}`eq-wm-fresnel-p` deserves its own exercise. The numerator of $r_p$ is
$n_2\cos\theta_i-n_1\cos\theta_t$, and there is an angle where it vanishes outright: the
p-polarized reflection *disappears*. Combining that condition with Snell's law
{eq}`eq-wm-snell` gives **Brewster's angle** {eq}`eq-wm-brewster`,
$\tan\theta_B=n_2/n_1$, equivalently $\theta_B+\theta_t=90^\circ$.

The second form is the one to keep, because it says why ({numref}`fig-wm-brewster-setup`).
The reflected wave is not a wave that bounced; it is radiation emitted by the charges the
transmitted wave drives in the second medium. For p-polarization those charges oscillate
in the plane of incidence, perpendicular to the refracted ray, so when
$\theta_B+\theta_t=90^\circ$ the direction the reflected ray would take is *along* the
dipoles' own axis. A dipole radiates as $\sin^2\Theta$ from its axis
([§3.10](radiation.ipynb)) and therefore radiates nothing there at all. The reflection has
no source. Light reflected off water or glass near $\theta_B$ is consequently pure
s-polarization, which is exactly what a polarizing filter is turned to block.

The interface is **air to glass**, $n_1=1$, $n_2=1.5$, on the coefficients built in
Exercise 7.

**Part a)** Locate the zero of $\operatorname{Re}r_p(\theta_i)$ with
`scipy.optimize.brentq` bracketed on $[30^\circ,80^\circ]$, using the `fresnel_p` you
wrote in Exercise 7, and compare with $\arctan(n_2/n_1)=56.310^\circ$.

**Part b)** Compute $\theta_t$ at that angle from {eq}`eq-wm-snell` and confirm
$\theta_B+\theta_t=90^\circ$.

**Part c)** Form the degree of polarization of the reflected light,
$P=(R_s-R_p)/(R_s+R_p)$, on the angle grid of Exercise 7, plot it
({numref}`fig-wm-brewster`), and confirm $P=1$ at $\theta_B$: at that one angle the
reflection is completely polarized.

**Part d)** Repeat Part a for **glass to air** ($n_1=1.5$, $n_2=1$) and check that the two
Brewster angles are complementary, $\arctan(1/1.5)=33.690^\circ=90^\circ-56.310^\circ$,
which is the same geometric statement seen from the other side of the glass.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    th_B_num,
    TH_B,
    "the zero of r_p sits at Brewster's angle arctan(n₂/n₁) = 56.310°",
    rtol=1e-9,
)
validate.close(
    th_B_num + th_t_at_B,
    np.pi / 2.0,
    "there the reflected and refracted rays are exactly perpendicular, θ_B + θ_t = 90°",
    rtol=1e-9,
)
validate.close(
    P_at_B,
    1.0,
    "so the reflected light is completely s-polarized: the degree of polarization is 1",
    rtol=1e-4,
)
validate.close(
    th_B_rev,
    np.arctan(N_AIR / N_GLASS),
    "and the glass-to-air Brewster angle is its complement, 33.690°",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — Total internal reflection and the evanescent wave (student)

Send the light the other way, from glass into air, and Exercise 6 already showed what
goes wrong: past $\theta_c=\arcsin(n_2/n_1)=41.810^\circ$ the phase-matching condition
{eq}`eq-wm-snell` has no real solution. Nothing breaks, though. Write
$\cos\theta_t=\sqrt{1-(n_1\sin\theta_i/n_2)^2}$ as a complex number and it becomes
$i\,q$ with $q=\sqrt{(n_1\sin\theta_i/n_2)^2-1}$ real, the Fresnel formulas
{eq}`eq-wm-fresnel-s` and {eq}`eq-wm-fresnel-p` carry straight through, and the
transmitted wave turns into the **evanescent** field {eq}`eq-wm-evanescent`: a
disturbance that runs *along* the surface with the incident wave's tangential wavevector
and dies exponentially away from it over the penetration depth
$d=\lambda/2\pi\sqrt{n_1^2\sin^2\theta_i-n_2^2}$ ({numref}`fig-wm-tir-setup`).

Three things about that field are worth measuring rather than believing. Both reflection
coefficients acquire modulus exactly one, so the reflection is *total* — better than any
metal mirror, which is why an optical fibre is glass and not silver. Each acquires a
phase, and the phases differ between the two polarizations, which is how a glass prism
with no coating on it can be made into a quarter-wave plate. And the field on the far
side is not small: at $45^\circ$ its amplitude *exceeds* the incident one, while the
time-averaged energy flux normal to the interface, $\langle S_z\rangle=|E_t|^2
\operatorname{Re}\tilde k_z/2\mu_0\omega$, is exactly zero because $\tilde k_z$ is purely
imaginary. A large field that transports nothing across the boundary is what "total"
means. Bring a second glass surface within a few hundred nanometres and the decay is cut
short before it finishes: light crosses the gap, an optical tunnelling that is the
working principle of the fingerprint sensor on a phone.

The interface is **glass to air**, $n_1=1.5$, $n_2=1$, at $\lambda=633\,$nm, using the
`fresnel_s` and `fresnel_p` you built in Exercise 7, which already keep $\cos\theta_t$
complex.

```{admonition} With your assistant
:class: tip
Part e wants an animation of the field on both sides of the interface: two half-spaces
on one grid, a frame index, an `imshow` whose array is refreshed each frame. That
scaffolding is generic and a reasonable thing to have generated for you. The physics
inside it is not. Check yourself that the field your animation draws above the interface
decays with the penetration depth $d$ of {eq}`eq-wm-evanescent`, by fitting $\ln|E|$
against $z$ with `numpy.polyfit` and comparing the slope with $-1/d$. The check is yours.
```

**Part a)** Evaluate `fresnel_s` and `fresnel_p` on 801 angles spaced linearly over
$[\theta_c+0.01^\circ,\,89.9^\circ]$ and confirm with `numpy.allclose` that
$|r_s|=|r_p|=1$ to $10^{-12}$: past the critical angle no power at all crosses the
boundary.

**Part b)** Take the reflection phases with `numpy.angle` and compare them with the
closed forms $\delta_s=-2\arctan(q'/\cos\theta_i)$ and
$\delta_p=-2\arctan\bigl((n_1/n_2)^2q'/\cos\theta_i\bigr)$, where
$q'=\sqrt{\sin^2\theta_i-(n_2/n_1)^2}$. Then find the angle at which the *difference*
$\delta_p-\delta_s$ is largest, with `numpy.argmax` over the grid. It peaks just above
$45^\circ$ at slightly more than $45^\circ$ of phase difference, which is why two
internal reflections in a suitably cut glass block (a Fresnel rhomb) turn linear
polarization into circular with no coating at all.

**Part c)** Compute the penetration depth $d$ of {eq}`eq-wm-evanescent` across the same
grid, and report it at $45^\circ$ and $60^\circ$.

**Part d)** At $\theta_i=45^\circ$ report $|t_s|^2$, and evaluate the normal energy flux
$\langle S_z\rangle\propto\operatorname{Re}\tilde k_z$ with $\tilde k_z=(2\pi
n_2/\lambda)\cos\theta_t$, together with the along-surface flux
$\langle S_x\rangle\propto\operatorname{Re}\tilde k_x$. The first is zero and the second
is not: the evanescent field carries energy *along* the interface and none across it.

**Part e)** Animate the total field on a grid spanning $x\in[0,3]\,\mu$m and
$z\in[-1.2,0.9]\,\mu$m over one optical period ({numref}`fig-wm-evanescent-anim`): in the
glass the sum $e^{i(k_xx+k_zz-\omega t)}+r_se^{i(k_xx-k_zz-\omega t)}$, in the air the
evanescent $t_se^{ik_xx}e^{-z/d}e^{-i\omega t}$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    total_s and total_p,
    "beyond the critical angle both Fresnel coefficients have modulus one: the "
    "reflection is total for either polarization",
)
validate.check(
    phases_ok, "and their phases match the closed-form total-reflection phases"
)
validate.close(
    D45,
    284.7e-9,
    "the evanescent field penetrates 285 nm into the air at 45° for λ = 633 nm",
    rtol=1e-3,
)
validate.close(
    S_z_factor,
    0.0,
    "yet it carries no energy across the boundary: Re k̃_z = 0, so ⟨S_z⟩ = 0",
    atol=1e-12,
)
validate.check(
    abs(t_s_45) ** 2 > 1.0,
    "while the transmitted amplitude exceeds the incident one",
    f"|t_s|² = {abs(t_s_45) ** 2:.4f}",
)

In [ ]:
# (solution hidden on the public site)


## Notebook summary

- **One electron on a spring is the whole of optics.** The driven Lorentz oscillator
  {eq}`eq-wm-eom` integrated with `DOP853` reproduced its closed-form complex amplitude
  {eq}`eq-wm-xamp` to one part in $10^6$ across fifteen drive frequencies, with the lag
  passing through exactly $\pi/2$ on resonance (Exercise 1). It is the same equation, and
  the same resonance curve, as the damped driven pendulum of
  [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb).
- **The dielectric function** {eq}`eq-wm-epsilon`: with $\hbar\omega_0=10\,$eV and
  $n_b=9.07\times10^{28}\,\mathrm{m^{-3}}$ the static value came out $2.2506$
  ($n=1.500$, ordinary glass), $\operatorname{Im}\varepsilon_r$ was positive at every
  frequency, and $\operatorname{Re}\varepsilon_r$ crossed zero at $14.99\,$eV against the
  predicted $\hbar\sqrt{\omega_0^2+\omega_p^2}=15.00\,$eV, bounding a $10.0$–$15.0\,$eV
  band in which nothing propagates (Exercise 2).
- **Refraction and absorption are one square root** {eq}`eq-wm-index`: $n^2-\kappa^2=
  \operatorname{Re}\varepsilon_r$ and $2n\kappa=\operatorname{Im}\varepsilon_r$ held to
  machine precision, the absorption length $1/\alpha$ ran from $11\,\mu$m at $2\,$eV to
  $2.8\,$nm on resonance, and $dn/d\omega<0$ on one contiguous band, $9.85$–$13.77\,$eV,
  containing the absorption peak: anomalous dispersion lives inside the line and nowhere
  else (Exercise 3).
- **The conductor is the unbound limit** {eq}`eq-wm-drude`: copper's
  $\sigma=ne^2\tau/m=5.967\times10^7\,$S/m matched the measured $5.96\times10^7\,$S/m,
  and the full Drude skin depth agreed with $\sqrt{2/\mu_0\sigma\omega}$ to eleven
  digits at $50\,$Hz and $1\,$MHz — $9.21\,$mm and $65.2\,\mu$m — parting from it only
  above $10^{11}\,$Hz, and wrong by a factor $6.2$ at $633\,$nm (Exercise 4).
- **The plasma frequency** {eq}`eq-wm-plasma`: aluminium's $\hbar\omega_p=15.79\,$eV
  against the measured $15.8\,$eV, with $R=1$ to $10^{-12}$ at every frequency below it
  in the collisionless limit, and the ionosphere's $f_p=2.839\,$MHz agreeing with the
  radio rule $8.98\sqrt{n_e}$ — so $\varepsilon_r=-7.06$ for an AM carrier (reflected)
  and $+0.999$ for an FM one (lost to space) (Exercise 5).
- **Snell's law is phase matching** {eq}`eq-wm-snell`: the interface mismatch was
  *exactly* zero at $\theta_t=25.374^\circ$ and of order $2$ two degrees either side,
  the reflected angle came back as $40.000000^\circ$, and glass into air at $60^\circ$
  left the mismatch positive across the entire bracket, with the critical angle at
  $41.810315^\circ$ (Exercise 6).
- **The Fresnel relations** {eq}`eq-wm-fresnel-s`, {eq}`eq-wm-fresnel-p` fell out of a
  $2\times2$ boundary-condition system solved with `numpy.linalg.solve`, matching the
  closed forms at every angle and giving $R+T=1$ to $7\times10^{-16}$ although nothing
  about energy went in, with the four-percent normal-incidence reflection of a window
  pane (Exercise 7).
- **Brewster's angle** {eq}`eq-wm-brewster` at $56.309932^\circ$, with
  $\theta_B+\theta_t=90.000000000^\circ$ and a reflected degree of polarization of $1$;
  the glass-to-air value is its complement, $33.690068^\circ$ (Exercise 8).
- **Total internal reflection** {eq}`eq-wm-evanescent`: $|r_s|=|r_p|=1$ to $10^{-12}$
  past $\theta_c$, phases matching their closed forms, a maximum polarization phase
  difference of $45.24^\circ$ near $51.7^\circ$ (the Fresnel rhomb), a penetration depth
  of $285\,$nm at $45^\circ$, and $|t_s|^2=3.6$ with $\operatorname{Re}\tilde k_z=0$: a
  field larger than the incident one that carries exactly no energy across the boundary
  (Exercise 9).

## Outlook

- **Causality ties the two halves of $\varepsilon$ together.** We found absorption and
  anomalous dispersion peaking on the same line and left it at that. It is not a
  coincidence: because a medium cannot respond before it is driven, $\varepsilon(\omega)$
  is analytic in the upper half plane, and $\operatorname{Re}\varepsilon$ is then
  determined by $\operatorname{Im}\varepsilon$ at all other frequencies through the
  Kramers–Kronig relations. Those are derived, and verified on this very Lorentz
  susceptibility, in
  [§7.2](../07-quantum-statistical-mechanics/complex-analysis-applications.ipynb).
- **Anisotropic media**, and this one is delivered rather than deferred. We assumed the
  response was a scalar. In a crystal it is a
  tensor: $\mathbf D=\varepsilon_0\boldsymbol\varepsilon\cdot\mathbf E$, the index depends
  on the direction of polarization, and one ray entering calcite becomes two.
  [§3.16](anisotropic-dielectrics.ipynb) takes the permittivity apart as a property
  tensor — how it transforms, what survives rotation, and why $\mathbf D$ stops pointing
  along $\mathbf E$ — and [§3.17](crystal-optics.ipynb) then puts light through it:
  birefringence, optic axes, the walk-off that sends the ray somewhere the wave normal
  is not, and the wave plates that follow from it.
- **Where $\varepsilon$ really comes from.** The oscillator strength and resonance
  frequency were handed to us as parameters. In a real solid they are the interband
  transitions of the electronic structure, and computing $\varepsilon_2(\omega)$ from
  dipole matrix elements and a joint density of states is the subject of
  [§8.15](../08-electronic-structure/optics-excitons.ipynb) — the same curve, earned
  rather than modelled.
- **Guided and radiated waves.** Total internal reflection at two parallel surfaces is an
  optical fibre, and the perfect metallic reflection of Exercise 5 is what makes the
  conducting walls of [§3.9](waveguides-cavities.ipynb) work; the dipole radiation
  pattern that killed the Brewster reflection is
  [§3.10](radiation.ipynb).
- **Beyond linear response.** Everything here assumed $P\propto E$. Drive the oscillator
  {eq}`eq-wm-eom` hard enough to feel the anharmonicity of a real binding potential and
  the polarization acquires terms in $E^2$ and $E^3$: frequency doubling, self-focusing,
  and the whole of nonlinear optics, which this course names as a horizon and does not
  develop.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()